In [1]:
%%capture
!pip install -U langgraph langchain langchain-core pydantic-settings
!pip    -U qdrant-client sentence-transformers rank-bm25 requests

In [2]:
# ============================================================
# CELL 2: Logging setup — single shared logger for the whole harness
# ============================================================
import logging
import sys

def setup_logger(name: str = "rag_harness") -> logging.Logger:
    logger = logging.getLogger(name)
    if logger.handlers:  # avoid duplicate handlers on Colab re-run
        return logger
    logger.setLevel(logging.INFO)
    handler = logging.StreamHandler(sys.stdout)
    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%H:%M:%S",
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    return logger

logger = setup_logger()
logger.info("Logger initialized")

17:47:23 | INFO     | rag_harness | Logger initialized


INFO:rag_harness:Logger initialized


In [ ]:
# ============================================================
# CELL 3: Settings (your pattern, extended for the full harness)
# ============================================================
from pydantic_settings import BaseSettings
from functools import lru_cache


class Settings(BaseSettings):
    # ---- (primary LLM, text-only agents) ----
    MODEL_NAME: str = ""
    API_URL: str = ""
    AUTH_TOKEN: str = ""
    TIMEOUT_SECONDS: int = 60
    MAX_RETRIES: int = 3
    RETRY_DELAY_SECONDS: int = 5

    # ---- OpenRouter (fallback / secondary LLM) ----
    OPENROUTER_API_KEY: str = "REPLACE_ME"
    OPENROUTER_MODEL: str = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free"
    OPENROUTER_MAX_TOKENS: int = 2000
    OPENROUTER_TEMPERATURE: float = 0.0
    OPENROUTER_BASE_URL: str = "https://openrouter.ai/api/v1"

    # ---- Qdrant ----
    QDRANT_URL: str = "http://localhost:6333"
    QDRANT_API_KEY: str = ""
    QDRANT_COLLECTION: str = "rag_harness_chunks"
    EMBEDDING_DIM: int = 384  # matches all-MiniLM-L6-v2

    # ---- Embedding / rerank / NLI models (all local, HF) ----
    EMBEDDING_MODEL: str = "sentence-transformers/all-MiniLM-L6-v2"
    RERANK_MODEL: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    NLI_MODEL: str = "cross-encoder/nli-deberta-v3-base"

    # ---- Harness loop control ----
    MAX_ITERATIONS: int = 8
    MAX_SEARCH_ATTEMPTS: int = 4
    MAX_TOKEN_BUDGET: int = 12000
    MAX_VALIDATION_RETRIES: int = 2

    # ---- Retrieval tuning ----
    RRF_K: int = 60
    TOP_K_DENSE: int = 20
    TOP_K_SPARSE: int = 20
    TOP_K_OKF: int = 10
    TOP_K_FINAL_RERANK: int = 8
    DEDUP_SIMILARITY_THRESHOLD: float = 0.92
    MIN_RELEVANCE_SCORE: float = 0.35  # below this -> "insufficient information"

    # ---- NLI groundedness thresholds ----
    NLI_ENTAILMENT_THRESHOLD: float = 0.6
    NLI_CONTRADICTION_MAX: float = 0.3

    class Config:
        env_file = ".env"
        extra = "ignore"


@lru_cache
def get_settings() -> Settings:
    logger.info("Settings loaded")
    return Settings()


settings = get_settings()

17:47:26 | INFO     | rag_harness | Settings loaded


/tmp/ipykernel_3202/1125646558.py:8: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class Settings(BaseSettings):
INFO:rag_harness:Settings loaded


In [4]:
# ============================================================
# CELL 4: Custom exceptions — explicit failure taxonomy
# ============================================================

class HarnessError(Exception):
    """Base class for all harness-level errors."""


class LLMClientError(HarnessError):
    """Raised when the LLM backend fails after all retries."""


class IngestionError(HarnessError):
    """Raised when a document fails to parse/route during ingestion."""


class RetrievalError(HarnessError):
    """Raised when retrieval pipeline fails (vector DB down, empty index, etc)."""


class ValidationError(HarnessError):
    """Raised when citation validation hard-fails (hallucinated chunk_id)."""


class LoopBudgetExceeded(HarnessError):
    """Raised (and caught internally) when iteration/token/search caps are hit."""

In [5]:
# ============================================================
# CELL 5: LLM Client (your pattern) + Factory for multi-backend support
# ============================================================
import requests
import time
from abc import ABC, abstractmethod


class BaseLLMClient(ABC):
    @abstractmethod
    def generate(
        self,
        system_prompt: str,
        user_content: str,
        response_format: dict = None,
        max_tokens: int = 500,
        temperature: float = 0.0,
    ) -> str:
        ...


class MedhaLLMClient(BaseLLMClient):
    """Wraps calls to the self-hosted Medha endpoint. Text-only agents."""

    def __init__(self, settings: Settings):
        self.settings = settings
        self.logger = logging.getLogger("rag_harness.llm.medha")

    def generate(
        self,
        system_prompt: str,
        user_content: str,
        response_format: dict = None,
        max_tokens: int = 500,
        temperature: float = 0.0,
    ) -> str:
        payload = {
            "model": self.settings.MODEL_NAME,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content},
            ],
            "temperature": temperature,
            "max_tokens": max_tokens,
        }
        if response_format is not None:
            payload["response_format"] = response_format

        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.settings.AUTH_TOKEN}",
        }

        last_error = None
        for attempt in range(1, self.settings.MAX_RETRIES + 1):
            try:
                self.logger.info(f"Calling Medha (attempt {attempt}/{self.settings.MAX_RETRIES})")
                resp = requests.post(
                    self.settings.API_URL, headers=headers, json=payload,
                    timeout=self.settings.TIMEOUT_SECONDS,
                )
                if resp.status_code >= 400:
                    last_error = f"http_{resp.status_code}: {resp.text[:1000]}"
                    self.logger.warning(f"Medha returned error: {last_error}")
                    if 400 <= resp.status_code < 500:
                        break  # client error, retrying won't help
                    if attempt < self.settings.MAX_RETRIES:
                        time.sleep(self.settings.RETRY_DELAY_SECONDS)
                    continue
                resp.raise_for_status()
                result = resp.json()
                return result["choices"][0]["message"]["content"]
            except requests.exceptions.RequestException as e:
                last_error = f"request_failed: {str(e)}"
                self.logger.warning(last_error)
                if attempt < self.settings.MAX_RETRIES:
                    time.sleep(self.settings.RETRY_DELAY_SECONDS)
            except (KeyError, IndexError) as e:
                last_error = f"unexpected_response_shape: {str(e)}"
                self.logger.error(last_error)
                if attempt < self.settings.MAX_RETRIES:
                    time.sleep(self.settings.RETRY_DELAY_SECONDS)

        self.logger.error(f"Giving up after {self.settings.MAX_RETRIES} attempts: {last_error}")
        raise LLMClientError(f"MedhaLLMClient giving up: {last_error}")


class OpenRouterLLMClient(BaseLLMClient):
    """Fallback backend via OpenRouter — same interface as MedhaLLMClient."""

    def __init__(self, settings: Settings):
        self.settings = settings
        self.logger = logging.getLogger("rag_harness.llm.openrouter")

    def generate(
        self,
        system_prompt: str,
        user_content: str,
        response_format: dict = None,
        max_tokens: int = None,
        temperature: float = None,
    ) -> str:
        payload = {
            "model": self.settings.OPENROUTER_MODEL,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content},
            ],
            "temperature": temperature if temperature is not None else self.settings.OPENROUTER_TEMPERATURE,
            "max_tokens": max_tokens or self.settings.OPENROUTER_MAX_TOKENS,
        }
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.settings.OPENROUTER_API_KEY}",
        }
        last_error = None
        for attempt in range(1, self.settings.MAX_RETRIES + 1):
            try:
                self.logger.info(f"Calling OpenRouter (attempt {attempt}/{self.settings.MAX_RETRIES})")
                resp = requests.post(
                    f"{self.settings.OPENROUTER_BASE_URL}/chat/completions",
                    headers=headers, json=payload, timeout=self.settings.TIMEOUT_SECONDS,
                )
                resp.raise_for_status()
                result = resp.json()
                return result["choices"][0]["message"]["content"]
            except requests.exceptions.RequestException as e:
                last_error = f"request_failed: {str(e)}"
                self.logger.warning(last_error)
                if attempt < self.settings.MAX_RETRIES:
                    time.sleep(self.settings.RETRY_DELAY_SECONDS)
            except (KeyError, IndexError) as e:
                last_error = f"unexpected_response_shape: {str(e)}"
                self.logger.error(last_error)
                break

        raise LLMClientError(f"OpenRouterLLMClient giving up: {last_error}")


class LLMClientFactory:
    """Factory method — pick backend by name, keep call sites decoupled from concrete classes."""

    _registry = {
        "medha": MedhaLLMClient,
        "openrouter": OpenRouterLLMClient,
    }

    @classmethod
    def create(cls, backend: str, settings: Settings) -> BaseLLMClient:
        if backend not in cls._registry:
            raise ValueError(f"Unknown LLM backend '{backend}'. Available: {list(cls._registry)}")
        logger.info(f"Instantiating LLM backend: {backend}")
        return cls._registry[backend](settings)


# Default client for the whole harness
llm_client: BaseLLMClient = LLMClientFactory.create("medha", settings)

17:47:29 | INFO     | rag_harness | Instantiating LLM backend: medha


INFO:rag_harness:Instantiating LLM backend: medha


In [6]:
# ============================================================
# CELL 6: XGrammar (your pattern) — structured output enforcement
# ============================================================
from pydantic import BaseModel


class XGrammar:
    """Wraps a Pydantic model into a strict JSON-schema response_format."""

    def __init__(self, pydantic_model: type[BaseModel], name: str) -> None:
        self.pydantic_model = pydantic_model
        self.name = name

    def build_response_format(self) -> dict:
        return {
            "type": "json_schema",
            "json_schema": {
                "name": self.name,
                "strict": True,
                "schema": self.pydantic_model.model_json_schema(),
            },
        }

In [7]:
# ============================================================
# CELL 7: Embedding + NLI + Rerank models — loaded once, local, no API cost
# ============================================================
from sentence_transformers import SentenceTransformer, CrossEncoder


class ModelRegistry:
    """Lazy-loads and caches local models so we don't reload per call."""

    _embedder = None
    _reranker = None
    _nli_model = None

    @classmethod
    def embedder(cls) -> SentenceTransformer:
        if cls._embedder is None:
            logger.info(f"Loading embedding model: {settings.EMBEDDING_MODEL}")
            cls._embedder = SentenceTransformer(settings.EMBEDDING_MODEL)
        return cls._embedder

    @classmethod
    def reranker(cls) -> CrossEncoder:
        if cls._reranker is None:
            logger.info(f"Loading rerank model: {settings.RERANK_MODEL}")
            cls._reranker = CrossEncoder(settings.RERANK_MODEL)
        return cls._reranker

    @classmethod
    def nli_model(cls) -> CrossEncoder:
        if cls._nli_model is None:
            logger.info(f"Loading NLI model: {settings.NLI_MODEL}")
            cls._nli_model = CrossEncoder(settings.NLI_MODEL)
        return cls._nli_model

In [8]:
# ============================================================
# CELL 8: Data models — chunk, retrieval result, claim, state
# ============================================================
from pydantic import BaseModel, Field
from typing import Optional, Literal
import uuid


class ChunkMetadata(BaseModel):
    doc_id: str
    doc_version: str
    source_type: Literal["pdf", "md"]
    page_num: Optional[int] = None
    section_title: Optional[str] = None
    ingestion_route: Optional[str] = None  # "anydoc" or "docling", pdf only


class Chunk(BaseModel):
    chunk_id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    raw_text: str
    contextual_text: str  # raw_text prefixed with LLM-generated context preamble
    metadata: ChunkMetadata
    embedding: Optional[list[float]] = None


class RetrievedChunk(BaseModel):
    chunk: Chunk
    score: float
    source: Literal["dense", "sparse", "okf", "fused"]


class Claim(BaseModel):
    claim_text: str
    cited_chunk_ids: list[str]


class GroundednessResult(BaseModel):
    claim: Claim
    entailment_score: float
    contradiction_score: float
    is_grounded: bool
    reason: Optional[str] = None

In [9]:
# ============================================================
# CELL 9: Qdrant vector store wrapper
# ============================================================
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue


class VectorStore:
    def __init__(self, settings: Settings):
        self.settings = settings
        self.logger = logging.getLogger("rag_harness.vectorstore")
        self.client = QdrantClient(url=settings.QDRANT_URL, api_key=settings.QDRANT_API_KEY or None)
        self._ensure_collection()

    def _ensure_collection(self):
        try:
            collections = [c.name for c in self.client.get_collections().collections]
            if self.settings.QDRANT_COLLECTION not in collections:
                self.logger.info(f"Creating Qdrant collection: {self.settings.QDRANT_COLLECTION}")
                self.client.create_collection(
                    collection_name=self.settings.QDRANT_COLLECTION,
                    vectors_config=VectorParams(size=self.settings.EMBEDDING_DIM, distance=Distance.COSINE),
                )
        except Exception as e:
            self.logger.error(f"Failed to ensure collection: {e}")
            raise RetrievalError(f"Qdrant collection setup failed: {e}")

    def upsert_chunk(self, chunk: Chunk):
        try:
            self.client.upsert(
                collection_name=self.settings.QDRANT_COLLECTION,
                points=[PointStruct(
                    id=chunk.chunk_id,
                    vector=chunk.embedding,
                    payload={
                        "raw_text": chunk.raw_text,
                        "contextual_text": chunk.contextual_text,
                        **chunk.metadata.model_dump(),
                    },
                )],
            )
        except Exception as e:
            self.logger.error(f"Upsert failed for chunk {chunk.chunk_id}: {e}")
            raise IngestionError(f"Qdrant upsert failed: {e}")

    def dense_search(self, query_vector: list[float], top_k: int, filters: dict = None) -> list[RetrievedChunk]:
        try:
            qdrant_filter = self._build_filter(filters) if filters else None
            hits = self.client.search(
                collection_name=self.settings.QDRANT_COLLECTION,
                query_vector=query_vector,
                limit=top_k,
                query_filter=qdrant_filter,
            )
            return [self._hit_to_retrieved_chunk(h, "dense") for h in hits]
        except Exception as e:
            self.logger.error(f"Dense search failed: {e}")
            raise RetrievalError(f"Dense search failed: {e}")

    def _build_filter(self, filters: dict) -> Filter:
        conditions = [FieldCondition(key=k, match=MatchValue(value=v)) for k, v in filters.items()]
        return Filter(must=conditions)

    def _hit_to_retrieved_chunk(self, hit, source: str) -> RetrievedChunk:
        payload = hit.payload
        chunk = Chunk(
            chunk_id=str(hit.id),
            raw_text=payload["raw_text"],
            contextual_text=payload["contextual_text"],
            metadata=ChunkMetadata(**{k: v for k, v in payload.items() if k in ChunkMetadata.model_fields}),
        )
        return RetrievedChunk(chunk=chunk, score=hit.score, source=source)


vector_store = VectorStore(settings)

ModuleNotFoundError: No module named 'qdrant_client'

In [ ]:
# ============================================================
# CELL 10: Sparse (BM25) retriever
# ============================================================
from rank_bm25 import BM25Okapi


class SparseRetriever:
    """In-memory BM25 index. For production scale, swap for Elasticsearch/Opensearch,
    but the interface below stays the same."""

    def __init__(self):
        self.logger = logging.getLogger("rag_harness.sparse")
        self.corpus_chunks: list[Chunk] = []
        self.bm25: Optional[BM25Okapi] = None

    def index(self, chunks: list[Chunk]):
        self.corpus_chunks = chunks
        tokenized = [c.contextual_text.lower().split() for c in chunks]
        self.bm25 = BM25Okapi(tokenized)
        self.logger.info(f"Indexed {len(chunks)} chunks into BM25")

    def search(self, query: str, top_k: int) -> list[RetrievedChunk]:
        if self.bm25 is None:
            raise RetrievalError("BM25 index not built — call .index() first")
        tokenized_query = query.lower().split()
        scores = self.bm25.get_scores(tokenized_query)
        ranked = sorted(zip(self.corpus_chunks, scores), key=lambda x: x[1], reverse=True)[:top_k]
        return [RetrievedChunk(chunk=c, score=float(s), source="sparse") for c, s in ranked if s > 0]


sparse_retriever = SparseRetriever()

In [ ]:
# ============================================================
# CELL 11: OKF (Open Knowledge Format / Google) retriever stub
# ============================================================
class OKFRetriever:
    """Retrieval-only interface into an OKF-formatted external knowledge source.
    NOT a live web search — OKF is a structured knowledge dataset you'd index
    the same way as your own corpus (embed + store), then query like any other source.
    Replace _fetch_okf_records with your actual OKF data access layer."""

    def __init__(self, settings: Settings):
        self.settings = settings
        self.logger = logging.getLogger("rag_harness.okf")

    def search(self, query: str, top_k: int) -> list[RetrievedChunk]:
        try:
            records = self._fetch_okf_records(query, top_k)
            return [
                RetrievedChunk(
                    chunk=Chunk(
                        chunk_id=r["id"],
                        raw_text=r["text"],
                        contextual_text=r["text"],
                        metadata=ChunkMetadata(
                            doc_id=r.get("doc_id", "okf_external"),
                            doc_version="okf_v1",
                            source_type="md",
                        ),
                    ),
                    score=r["score"],
                    source="okf",
                )
                for r in records
            ]
        except Exception as e:
            self.logger.warning(f"OKF retrieval failed, degrading gracefully (no OKF results): {e}")
            return []  # OKF is a secondary signal — never hard-fail the whole retrieval on it

    def _fetch_okf_records(self, query: str, top_k: int) -> list[dict]:
        # STUB: replace with actual OKF dataset query (e.g. indexed OKF triples/passages).
        raise NotImplementedError("Wire this to your actual OKF data source")


okf_retriever = OKFRetriever(settings)

In [ ]:
# ============================================================
# CELL 12: RRF fusion (two-stage, per our design decision) + dedup + rerank
# ============================================================
import numpy as np


class RetrievalFuser:
    def __init__(self, settings: Settings):
        self.settings = settings
        self.logger = logging.getLogger("rag_harness.fuser")

    def rrf_fuse(self, ranked_lists: list[list[RetrievedChunk]], k: int = None) -> list[RetrievedChunk]:
        """Single RRF pass over N ranked lists. Rank-position based, score-scale-agnostic."""
        k = k or self.settings.RRF_K
        scores: dict[str, float] = {}
        chunk_map: dict[str, RetrievedChunk] = {}

        for ranked_list in ranked_lists:
            for rank, rc in enumerate(ranked_list, start=1):
                cid = rc.chunk.chunk_id
                scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank)
                chunk_map[cid] = rc  # keep last-seen chunk object

        fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return [
            RetrievedChunk(chunk=chunk_map[cid].chunk, score=score, source="fused")
            for cid, score in fused
        ]

    def two_stage_fuse(
        self, dense: list[RetrievedChunk], sparse: list[RetrievedChunk], okf: list[RetrievedChunk]
    ) -> list[RetrievedChunk]:
        """Stage 1: dense + sparse -> internal consensus.
        Stage 2: internal consensus + OKF -> final, with OKF as secondary signal.
        (Design decision: OKF should not equally outvote internal corpus agreement.)"""
        internal_fused = self.rrf_fuse([dense, sparse])
        final_fused = self.rrf_fuse([internal_fused, okf])
        self.logger.info(
            f"Two-stage RRF: {len(dense)} dense + {len(sparse)} sparse -> "
            f"{len(internal_fused)} internal, + {len(okf)} okf -> {len(final_fused)} final"
        )
        return final_fused

    def deduplicate(self, chunks: list[RetrievedChunk]) -> list[RetrievedChunk]:
        """Embedding-similarity based dedup — run BEFORE second RRF stage / before rerank."""
        if not chunks:
            return chunks
        embedder = ModelRegistry.embedder()
        texts = [c.chunk.contextual_text for c in chunks]
        embeddings = embedder.encode(texts, normalize_embeddings=True)

        keep_indices = []
        for i in range(len(chunks)):
            is_duplicate = False
            for j in keep_indices:
                sim = float(np.dot(embeddings[i], embeddings[j]))
                if sim >= self.settings.DEDUP_SIMILARITY_THRESHOLD:
                    is_duplicate = True
                    break
            if not is_duplicate:
                keep_indices.append(i)

        deduped = [chunks[i] for i in keep_indices]
        self.logger.info(f"Dedup: {len(chunks)} -> {len(deduped)} chunks")
        return deduped

    def rerank(self, query: str, chunks: list[RetrievedChunk], top_k: int = None) -> list[RetrievedChunk]:
        if not chunks:
            return chunks
        top_k = top_k or self.settings.TOP_K_FINAL_RERANK
        reranker = ModelRegistry.reranker()
        pairs = [(query, c.chunk.contextual_text) for c in chunks]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(chunks, scores), key=lambda x: x[1], reverse=True)[:top_k]
        return [RetrievedChunk(chunk=c.chunk, score=float(s), source=c.source) for c, s in ranked]


fuser = RetrievalFuser(settings)

In [ ]:
# ============================================================
# CELL 13: Full retrieval pipeline — orchestrates dense + sparse + okf + fuse + dedup + rerank
# ============================================================
class RetrievalPipeline:
    def __init__(self, vector_store: VectorStore, sparse: SparseRetriever, okf: OKFRetriever, fuser: RetrievalFuser, settings: Settings):
        self.vector_store = vector_store
        self.sparse = sparse
        self.okf = okf
        self.fuser = fuser
        self.settings = settings
        self.logger = logging.getLogger("rag_harness.retrieval_pipeline")

    def run(self, query: str, filters: dict = None) -> list[RetrievedChunk]:
        try:
            embedder = ModelRegistry.embedder()
            query_vector = embedder.encode(query, normalize_embeddings=True).tolist()

            dense_hits = self.vector_store.dense_search(query_vector, self.settings.TOP_K_DENSE, filters)
            sparse_hits = self.sparse.search(query, self.settings.TOP_K_SPARSE)
            okf_hits = self.okf.search(query, self.settings.TOP_K_OKF)

            fused = self.fuser.two_stage_fuse(dense_hits, sparse_hits, okf_hits)
            deduped = self.fuser.deduplicate(fused)
            reranked = self.fuser.rerank(query, deduped)

            self.logger.info(f"Retrieval complete: {len(reranked)} final chunks for query='{query[:50]}...'")
            return reranked
        except HarnessError:
            raise
        except Exception as e:
            self.logger.error(f"Retrieval pipeline failed unexpectedly: {e}")
            raise RetrievalError(f"Retrieval pipeline failed: {e}")


retrieval_pipeline = RetrievalPipeline(vector_store, sparse_retriever, okf_retriever, fuser, settings)

In [ ]:
# ============================================================
# CELL 14: NLI-based groundedness checker — runs on ALL claims, per your instruction
# ============================================================
class GroundednessChecker:
    def __init__(self, settings: Settings):
        self.settings = settings
        self.logger = logging.getLogger("rag_harness.groundedness")

    def check_claim(self, claim: Claim, chunk_registry: dict[str, Chunk]) -> GroundednessResult:
        cited_texts = []
        for cid in claim.cited_chunk_ids:
            if cid not in chunk_registry:
                # Should have been caught by citation validator already — defensive check here too
                self.logger.error(f"Claim cites unknown chunk_id={cid} — treat as ungrounded")
                return GroundednessResult(
                    claim=claim, entailment_score=0.0, contradiction_score=1.0,
                    is_grounded=False, reason="cited_chunk_not_in_registry",
                )
            cited_texts.append(chunk_registry[cid].raw_text)

        premise = " ".join(cited_texts)
        hypothesis = claim.claim_text

        try:
            nli_model = ModelRegistry.nli_model()
            # cross-encoder/nli-deberta-v3-base label order: [contradiction, entailment, neutral]
            scores = nli_model.predict([(premise, hypothesis)])[0]
            contradiction_score, entailment_score, neutral_score = (
                float(scores[0]), float(scores[1]), float(scores[2])
            )

            is_grounded = (
                entailment_score >= self.settings.NLI_ENTAILMENT_THRESHOLD
                and contradiction_score <= self.settings.NLI_CONTRADICTION_MAX
            )
            reason = None if is_grounded else (
                f"entailment={entailment_score:.2f} (need >= {self.settings.NLI_ENTAILMENT_THRESHOLD}), "
                f"contradiction={contradiction_score:.2f} (need <= {self.settings.NLI_CONTRADICTION_MAX})"
            )
            return GroundednessResult(
                claim=claim, entailment_score=entailment_score,
                contradiction_score=contradiction_score, is_grounded=is_grounded, reason=reason,
            )
        except Exception as e:
            self.logger.error(f"NLI check failed for claim '{hypothesis[:50]}...': {e}")
            # Fail closed: if the checker itself breaks, treat the claim as ungrounded
            return GroundednessResult(
                claim=claim, entailment_score=0.0, contradiction_score=1.0,
                is_grounded=False, reason=f"nli_check_error: {e}",
            )

    def check_all_claims(self, claims: list[Claim], chunk_registry: dict[str, Chunk]) -> list[GroundednessResult]:
        return [self.check_claim(c, chunk_registry) for c in claims]


groundedness_checker = GroundednessChecker(settings)

In [ ]:
# ============================================================
# CELL 15: Citation validator (code-only, cheapest check, runs first)
# ============================================================
class CitationValidator:
    def __init__(self):
        self.logger = logging.getLogger("rag_harness.citation_validator")

    def validate(self, claims: list[Claim], chunk_registry: dict[str, Chunk]) -> tuple[bool, list[str]]:
        """Returns (all_valid, list_of_hallucinated_chunk_ids)."""
        hallucinated = []
        for claim in claims:
            for cid in claim.cited_chunk_ids:
                if cid not in chunk_registry:
                    hallucinated.append(cid)
                    self.logger.warning(f"Hallucinated citation detected: {cid}")
        return (len(hallucinated) == 0, hallucinated)


citation_validator = CitationValidator()

In [ ]:
# ============================================================
# CELL 16: Simple caching layer (3-tier, in-memory dict for Colab;
# swap for Redis in production — interface stays identical)
# ============================================================
import hashlib


class CacheLayer:
    def __init__(self):
        self.logger = logging.getLogger("rag_harness.cache")
        self.query_cache: dict[str, str] = {}
        self.embedding_cache: dict[str, list[float]] = {}
        self.retrieval_cache: dict[str, list[RetrievedChunk]] = {}

    def _hash(self, text: str) -> str:
        return hashlib.sha256(text.encode()).hexdigest()

    def get_query_cache(self, query: str, doc_version_key: str) -> Optional[str]:
        key = self._hash(f"{query}::{doc_version_key}")
        hit = self.query_cache.get(key)
        if hit:
            self.logger.info("Query cache HIT")
        return hit

    def set_query_cache(self, query: str, doc_version_key: str, answer: str):
        key = self._hash(f"{query}::{doc_version_key}")
        self.query_cache[key] = answer

    def get_embedding_cache(self, text: str) -> Optional[list[float]]:
        return self.embedding_cache.get(self._hash(text))

    def set_embedding_cache(self, text: str, embedding: list[float]):
        self.embedding_cache[self._hash(text)] = embedding

    def invalidate_by_doc_version(self, doc_id: str, old_version: str):
        """Called on re-ingestion — selectively bust stale entries rather than wiping everything."""
        stale_keys = [k for k in self.query_cache if old_version in k]
        for k in stale_keys:
            del self.query_cache[k]
        self.logger.info(f"Invalidated {len(stale_keys)} stale query cache entries for doc {doc_id}")


cache_layer = CacheLayer()

In [ ]:
# ============================================================
# CELL 17: LangGraph agent state
# ============================================================
from typing import TypedDict, Annotated
import operator


class AgentState(TypedDict):
    query: str
    iteration: int
    max_iterations: int
    token_budget_used: int
    max_token_budget: int
    search_attempts: Annotated[list[str], operator.add]
    retrieved_chunk_registry: dict[str, Chunk]
    scratchpad: Annotated[list[str], operator.add]
    draft_answer: Optional[str]
    claims: list[Claim]
    citation_valid: bool
    groundedness_results: list[GroundednessResult]
    validation_retries: int
    final_answer: Optional[str]
    degraded: bool
    degrade_reason: Optional[str]

In [ ]:
# ============================================================
# CELL 18: Agent tools — plain functions the LangGraph nodes call
# ============================================================
class AgentTools:
    def __init__(self, retrieval_pipeline: RetrievalPipeline, vector_store: VectorStore):
        self.retrieval_pipeline = retrieval_pipeline
        self.vector_store = vector_store
        self.logger = logging.getLogger("rag_harness.tools")

    def search_documents(self, query: str, filters: dict = None) -> list[RetrievedChunk]:
        self.logger.info(f"TOOL search_documents(query='{query[:50]}...')")
        return self.retrieval_pipeline.run(query, filters)

    def search_documents_reformulated(self, original_query: str, failed_attempt_reason: str) -> list[RetrievedChunk]:
        self.logger.info(f"TOOL search_documents_reformulated(reason='{failed_attempt_reason}')")
        # The "reason" is logged for audit; reformulation itself happens upstream (agent writes new query)
        return self.retrieval_pipeline.run(original_query)

    def get_context(self, chunk_registry: dict[str, Chunk], chunk_id: str, window: int = 1) -> str:
        self.logger.info(f"TOOL get_context(chunk_id={chunk_id}, window={window})")
        if chunk_id not in chunk_registry:
            raise RetrievalError(f"chunk_id {chunk_id} not found in registry")
        return chunk_registry[chunk_id].raw_text  # simplified: real version fetches neighbors by page/order

    def get_page(self, doc_id: str, page_num: int) -> str:
        self.logger.info(f"TOOL get_page(doc_id={doc_id}, page_num={page_num})")
        try:
            hits = self.vector_store.client.scroll(
                collection_name=settings.QDRANT_COLLECTION,
                scroll_filter=Filter(must=[
                    FieldCondition(key="doc_id", match=MatchValue(value=doc_id)),
                    FieldCondition(key="page_num", match=MatchValue(value=page_num)),
                ]),
                limit=5,
            )
            points, _ = hits
            return "\n".join(p.payload["raw_text"] for p in points)
        except Exception as e:
            self.logger.error(f"get_page failed: {e}")
            raise RetrievalError(f"get_page failed: {e}")

    def check_document_freshness(self, doc_id: str) -> str:
        self.logger.info(f"TOOL check_document_freshness(doc_id={doc_id})")
        try:
            hits, _ = self.vector_store.client.scroll(
                collection_name=settings.QDRANT_COLLECTION,
                scroll_filter=Filter(must=[FieldCondition(key="doc_id", match=MatchValue(value=doc_id))]),
                limit=1,
            )
            if not hits:
                return "unknown"
            return hits[0].payload.get("doc_version", "unknown")
        except Exception as e:
            self.logger.error(f"check_document_freshness failed: {e}")
            return "unknown"


agent_tools = AgentTools(retrieval_pipeline, vector_store)

In [ ]:
# ============================================================
# CELL 19: LangGraph nodes — retrieval/reasoning node, generation node,
# verification node, degrade node
# ============================================================
from langgraph.graph import StateGraph, END
import re


def retrieve_node(state: AgentState) -> AgentState:
    logger.info(f"[retrieve_node] iteration={state['iteration']}")
    try:
        results = agent_tools.search_documents(state["query"])
        for rc in results:
            state["retrieved_chunk_registry"][rc.chunk.chunk_id] = rc.chunk
        state["search_attempts"] = [state["query"]]
        state["scratchpad"] = [f"Retrieved {len(results)} chunks for query."]
    except RetrievalError as e:
        logger.error(f"Retrieval failed in node: {e}")
        state["scratchpad"] = [f"Retrieval error: {e}"]
    state["iteration"] += 1
    return state


def generate_node(state: AgentState) -> AgentState:
    logger.info(f"[generate_node] iteration={state['iteration']}")
    context_blocks = []
    for cid, chunk in state["retrieved_chunk_registry"].items():
        context_blocks.append(
            f"[chunk_id={cid}] Document: {chunk.metadata.doc_id}, "
            f"Page: {chunk.metadata.page_num}\nContent: {chunk.raw_text}"
        )
    context_str = "\n\n".join(context_blocks)

    if not context_blocks:
        state["degraded"] = True
        state["degrade_reason"] = "no_chunks_retrieved"
        state["final_answer"] = "I don't have enough information to answer this question."
        return state

    system_prompt = (
        "You are a grounded QA assistant. Only use the provided context. "
        "Cite chunk_ids for every factual claim using the format [chunk_id=<id>]. "
        "If the context is insufficient, say so explicitly rather than guessing."
    )
    user_content = f"Context:\n{context_str}\n\nQuestion: {state['query']}\n\nAnswer:"

    try:
        answer = llm_client.generate(system_prompt, user_content, max_tokens=800)
        state["draft_answer"] = answer
        state["claims"] = _extract_claims(answer)
        state["token_budget_used"] += len(user_content.split())  # rough proxy, swap for real tokenizer
    except LLMClientError as e:
        logger.error(f"Generation failed: {e}")
        state["degraded"] = True
        state["degrade_reason"] = f"llm_generation_failed: {e}"
        state["final_answer"] = "I couldn't generate an answer due to a system error. Please try again."
    return state


def _extract_claims(answer_text: str) -> list[Claim]:
    """Splits answer into sentences, extracts cited chunk_ids per sentence via regex."""
    claims = []
    sentences = re.split(r"(?<=[.!?])\s+", answer_text)
    for sentence in sentences:
        cited_ids = re.findall(r"\[chunk_id=([\w-]+)\]", sentence)
        if cited_ids:
            claims.append(Claim(claim_text=sentence, cited_chunk_ids=cited_ids))
    return claims


def verify_node(state: AgentState) -> AgentState:
    logger.info(f"[verify_node] validation_retries={state['validation_retries']}")

    # Step 1: citation validation (cheap, code-only, runs first)
    is_valid, hallucinated = citation_validator.validate(state["claims"], state["retrieved_chunk_registry"])
    state["citation_valid"] = is_valid
    if not is_valid:
        logger.warning(f"Citation validation FAILED — hallucinated ids: {hallucinated}")
        state["validation_retries"] += 1
        state["scratchpad"] = [f"Citation validation failed: hallucinated chunk_ids {hallucinated}"]
        return state

    # Step 2: groundedness check on ALL claims (per your instruction, not scoped)
    results = groundedness_checker.check_all_claims(state["claims"], state["retrieved_chunk_registry"])
    state["groundedness_results"] = results
    ungrounded = [r for r in results if not r.is_grounded]

    if ungrounded:
        logger.warning(f"Groundedness FAILED for {len(ungrounded)} claim(s)")
        state["validation_retries"] += 1
        reasons = "; ".join(f"'{r.claim.claim_text[:50]}...' -> {r.reason}" for r in ungrounded)
        state["scratchpad"] = [f"Groundedness failed: {reasons}"]
    else:
        state["final_answer"] = state["draft_answer"]
        logger.info("Verification PASSED")

    return state


def degrade_node(state: AgentState) -> AgentState:
    logger.warning(f"[degrade_node] reason={state.get('degrade_reason')}")
    state["degraded"] = True
    if not state.get("final_answer"):
        # Strip ungrounded claims, keep the rest, per our repair-not-refuse design
        grounded_claims = [r.claim.claim_text for r in state.get("groundedness_results", []) if r.is_grounded]
        if grounded_claims:
            partial = " ".join(grounded_claims)
            state["final_answer"] = f"{partial}\n\n(Note: some claims could not be fully verified and were removed.)"
        else:
            state["final_answer"] = (
                "I couldn't produce a fully verified answer to this question. "
                "Please rephrase or consult the source documents directly."
            )
    return state

In [ ]:
# ============================================================
# CELL 20: LangGraph routing logic (edges) — this IS the loop control
# ============================================================
def route_after_retrieve(state: AgentState) -> str:
    if state["iteration"] >= state["max_iterations"]:
        state["degrade_reason"] = "max_iterations_exceeded"
        return "degrade"
    if state["token_budget_used"] >= state["max_token_budget"]:
        state["degrade_reason"] = "token_budget_exceeded"
        return "degrade"
    if len(state["search_attempts"]) >= settings.MAX_SEARCH_ATTEMPTS:
        state["degrade_reason"] = "max_search_attempts_exceeded"
        return "degrade"
    return "generate"


def route_after_verify(state: AgentState) -> str:
    if state.get("final_answer"):
        return "end"
    if state["validation_retries"] >= settings.MAX_VALIDATION_RETRIES:
        state["degrade_reason"] = "max_validation_retries_exceeded"
        return "degrade"
    if not state["citation_valid"]:
        return "generate"  # regenerate — hallucinated citation, no new retrieval needed
    return "generate"  # groundedness failure — regenerate more conservatively


def build_harness_graph():
    graph = StateGraph(AgentState)
    graph.add_node("retrieve", retrieve_node)
    graph.add_node("generate", generate_node)
    graph.add_node("verify", verify_node)
    graph.add_node("degrade", degrade_node)

    graph.set_entry_point("retrieve")
    graph.add_conditional_edges("retrieve", route_after_retrieve, {"generate": "generate", "degrade": "degrade"})
    graph.add_edge("generate", "verify")
    graph.add_conditional_edges("verify", route_after_verify, {"generate": "generate", "degrade": "degrade", "end": END})
    graph.add_edge("degrade", END)

    return graph.compile()


harness_graph = build_harness_graph()
logger.info("LangGraph harness compiled successfully")

In [ ]:
# ============================================================
# CELL 21: Entry point — run a query through the full harness
# ============================================================
def run_harness(query: str) -> str:
    logger.info(f"=== Running harness for query: '{query}' ===")
    initial_state: AgentState = {
        "query": query,
        "iteration": 0,
        "max_iterations": settings.MAX_ITERATIONS,
        "token_budget_used": 0,
        "max_token_budget": settings.MAX_TOKEN_BUDGET,
        "search_attempts": [],
        "retrieved_chunk_registry": {},
        "scratchpad": [],
        "draft_answer": None,
        "claims": [],
        "citation_valid": False,
        "groundedness_results": [],
        "validation_retries": 0,
        "final_answer": None,
        "degraded": False,
        "degrade_reason": None,
    }
    try:
        result = harness_graph.invoke(initial_state)
        logger.info(f"=== Harness complete. Degraded={result.get('degraded', False)} ===")
        return result["final_answer"]
    except HarnessError as e:
        logger.error(f"Harness failed with a known error: {e}")
        return "The system encountered an error and could not process your question. Please try again."
    except Exception as e:
        logger.critical(f"Harness failed with an UNEXPECTED error: {e}")
        return "An unexpected error occurred. This has been logged for review."



In [ ]:


answer = run_harness("What is the penalty clause for late payment under scheme X?")
print(answer)